# Notebook 27 — Where does the signal work?

**Purpose:** Diagnose whether pooled price/volume classifiers conceal ticker, sector, or regime-specific edges, and whether a cost-aware target is more useful. **Development-only:** the historical holdout and prospective ledger remain untouched.

**Important:** outer-fold segment/decile comparisons are exploratory; do not promote a subgroup or confidence threshold selected from these same folds. The economic binary target is a separate prediction task and its accuracy is **not directly comparable** with the Q7 three-class accuracy.


In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
if not (ROOT/'src').exists(): raise RuntimeError('Open notebook from repository root or notebooks directory')
if str(ROOT/'src') not in sys.path: sys.path.insert(0,str(ROOT/'src'))
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.decision_diagnostics import run_decision_diagnostics
import pandas as pd
import numpy as np
print('Project:',ROOT)


Project: /home/oem/Documents/github/stock_ml_dynamic


## 1. Research configuration

Six pooled configurations per fold (Random Forest, Logistic Regression, and ticker-aware Logistic Regression; each with price-only and price+volume). Additional sector-specific Logistic Regression experiments run **only when real sector metadata exists**. Three chronological folds, 10 confidence deciles per configuration, ticker and regime diagnostics, confusion matrices, and a separately labeled economic binary target.


In [2]:
N_FOLDS=3
MAX_TRAIN_ROWS=50000
TREE_TRAIN_ROWS=35000
COST_BPS=10
INCLUDE_VOLUME=True
OUTPUT=ROOT/'artifacts'/'decision_diagnostics'/'1d'
print('Expected pooled model fits:',N_FOLDS*6)


Expected pooled model fits: 18


## 2. Load frozen master data

Always unpack all four values returned by `load_master()`.


In [3]:
required=('master_labeled.parquet','latest_inference.parquet','transformers.joblib','manifest.json')
paths=[ROOT/'artifacts'/'master_tables'/'1d',ROOT/'notebooks'/'artifacts'/'master_tables'/'1d']
source=next((p for p in paths if all((p/f).exists() for f in required)),None)
if source is None: raise FileNotFoundError('Master artifacts missing; run Notebook 03')
master,latest,bundle,manifest=load_master(source)
print('Rows:',len(master),'symbols:',master.symbol.nunique(),'dates:',master.bar_timestamp.min(),master.bar_timestamp.max())
print('Available sector columns:',[c for c in ('sector','sector_name','gics_sector','sector_code','sector_id') if c in master])


Rows: 431271 symbols: 220 dates: 2018-11-23 00:00:00+00:00 2026-11-09 00:00:00+00:00
Available sector columns: []


## 3. Run research

Expect a longer execution time. Features, target quantiles, and preprocessing are fitted/constructed causally. The volatility-regime cutoff is estimated using each fold's training period only. Sector experiments are skipped if no trustworthy sector column is present.


In [4]:
outputs,cutoff,sector=run_decision_diagnostics(master,n_folds=N_FOLDS,
    max_train_rows=MAX_TRAIN_ROWS,tree_train_rows=TREE_TRAIN_ROWS,
    cost_bps=COST_BPS,include_volume=INCLUDE_VOLUME,output_dir=OUTPUT)
print('Development cutoff:',cutoff,'sector metadata:',sector or 'not available')
print('Saved CSV diagnostics to:',OUTPUT)


Development cutoff: 2025-04-07 00:00:00+00:00 sector metadata: not available
Saved CSV diagnostics to: /home/oem/Documents/github/stock_ml_dynamic/artifacts/decision_diagnostics/1d


## 4. Model performance: paired chronological folds

Use the same model and fold to compare price-only against price+volume. Examine fold-by-fold lift, not just averages.


In [5]:
scores=outputs['scores']
display(scores.sort_values(['model','fold']).round(4))
pooled=scores[scores.segment.eq('all')]
display(pooled.groupby('model').agg(accuracy=('accuracy','mean'),baseline=('majority_baseline','mean'),
    auc=('auc','mean'),entry_precision=('entry_precision','mean'),entry_coverage=('entry_coverage','mean'),
    entry_net=('entry_mean_net','mean'),folds=('fold','nunique')).sort_values('accuracy',ascending=False).round(4))


,fold,model,segment,segment_value,n,up_base,accuracy,majority_baseline,balanced_accuracy,auc,entry_count,entry_coverage,entry_precision,entry_mean_net
0,1,forest_price,all,all,51635,0.4218,0.4195,0.4218,0.3487,0.5508,6520,0.1263,0.4115,-0.0024
6,2,forest_price,all,all,51839,0.4117,0.4231,0.4117,0.3613,0.5572,1601,0.0309,0.4503,0.0012
12,3,forest_price,all,all,52247,0.4252,0.4279,0.4252,0.3591,0.5676,1677,0.0321,0.4770,-0.0023
3,1,forest_price_volume,all,all,51635,0.4218,0.4179,0.4218,0.3503,0.5574,6387,0.1237,0.4401,-0.0010
9,2,forest_price_volume,all,all,51839,0.4117,0.4215,0.4117,0.3599,0.5576,1764,0.0340,0.4580,0.0013
15,3,forest_price_volume,all,all,52247,0.4252,0.4269,0.4252,0.3563,0.5667,2660,0.0509,0.4951,0.0007
1,1,logistic_price,all,all,51635,0.4218,0.3981,0.4218,0.3362,0.5503,11783,0.2282,0.4546,0.0002
7,2,logistic_price,all,all,51839,0.4117,0.4181,0.4117,0.3638,0.5531,6313,0.1218,0.4480,0.0013
13,3,logistic_price,all,all,52247,0.4252,0.4133,0.4252,0.3571,0.5534,5066,0.0970,0.4388,-0.0022
4,1,logistic_price_volume,all,all,51635,0.4218,0.3946,0.4218,0.3402,0.5430,12958,0.2510,0.4546,-0.0007


,accuracy,baseline,auc,entry_precision,entry_coverage,entry_net,folds
model,,,,,,,
forest_price,0.4235,0.4196,0.5585,0.4463,0.0631,-0.0012,3
forest_price_volume,0.4221,0.4196,0.5605,0.4644,0.0695,0.0003,3
logistic_price,0.4098,0.4196,0.5523,0.4471,0.1490,-0.0002,3
logistic_price_volume,0.4078,0.4196,0.5489,0.4408,0.1684,-0.0005,3
logistic_ticker_price,0.4077,0.4196,0.5524,0.4499,0.1697,0.0000,3
logistic_ticker_price_volume,0.4063,0.4196,0.5527,0.4457,0.1868,-0.0001,3


## 5. Sector and ticker heterogeneity

Ticker rows are descriptive and have a minimum of 30 observations per fold. **Do not rank tickers for live trading from these results alone.** Sector-specific training requires an existing sector field; no mapping is fabricated.


In [6]:
display(outputs['ticker_diagnostics'].sort_values(['fold','entry_mean_net'],ascending=[True,False]).head(35).round(4))
if sector:
    display(scores[scores.model.eq('sector_logistic')].sort_values(['fold','segment_value']).round(4))
else:
    print('Sector-specific model skipped: master table has no sector metadata. Join point-in-time GICS reference in a future controlled experiment.')


,fold,model,segment,segment_value,n,up_base,accuracy,majority_baseline,balanced_accuracy,auc,entry_count,entry_coverage,entry_precision,entry_mean_net
182,1,forest_price,symbol,SW,227,0.2599,0.3392,0.5154,0.3444,0.5174,16,0.0705,0.3750,0.0108
127,1,forest_price,symbol,MNST,238,0.4328,0.4076,0.4328,0.3262,0.4781,14,0.0588,0.6429,0.0088
141,1,forest_price,symbol,NVDA,238,0.5084,0.4916,0.5084,0.3351,0.4898,64,0.2689,0.5469,0.0072
138,1,forest_price,symbol,NFLX,238,0.4496,0.4664,0.4496,0.3568,0.5247,52,0.2185,0.4808,0.0059
106,1,forest_price,symbol,KLAC,238,0.4118,0.3697,0.4118,0.3017,0.5021,50,0.2101,0.5200,0.0058
1,1,forest_price,symbol,ABBV,238,0.4160,0.4370,0.4160,0.3692,0.5449,10,0.0420,0.6000,0.0050
137,1,forest_price,symbol,NEM,238,0.3739,0.3782,0.4160,0.3337,0.4928,24,0.1008,0.4167,0.0047
163,1,forest_price,symbol,RBLX,238,0.4538,0.4454,0.4538,0.3317,0.4882,81,0.3403,0.4444,0.0043
31,1,forest_price,symbol,BX,238,0.4496,0.4412,0.4496,0.3302,0.4915,47,0.1975,0.4894,0.0042
116,1,forest_price,symbol,LRCX,238,0.4160,0.3992,0.4160,0.3209,0.4932,55,0.2311,0.4364,0.0039


Sector-specific model skipped: master table has no sector metadata. Join point-in-time GICS reference in a future controlled experiment.


## 6. Market regimes: does the edge persist?

High/low volatility uses a training-fold median. Breadth and momentum use current-bar signs. These are **diagnostic slices**, not strategies.


In [7]:
segments=outputs['segments']
for segment in ('volatility_regime','breadth_regime','momentum_regime'):
    print('\n',segment)
    display(segments[segments.segment.eq(segment)].groupby(['model','segment_value']).agg(
        n=('n','sum'),accuracy=('accuracy','mean'),entry_precision=('entry_precision','mean'),
        entry_coverage=('entry_coverage','mean'),entry_net=('entry_mean_net','mean')).round(4))



 volatility_regime


n  accuracy  entry_precision  \
model                        segment_value                                     
forest_price                 higher         70332    0.4346           0.4496   
                             lower          85389    0.4145           0.4442   
forest_price_volume          higher         70332    0.4327           0.4749   
                             lower          85389    0.4132           0.4386   
logistic_price               higher         70332    0.4207           0.4607   
                             lower          85389    0.4012           0.4270   
logistic_price_volume        higher         70332    0.4189           0.4561   
                             lower          85389    0.4000           0.4271   
logistic_ticker_price        higher         70332    0.4168           0.4613   
                             lower          85389    0.4005           0.4326   
logistic_ticker_price_volume higher         70332    0.4163           0.4566   
                             lower          85389    0.3983           0.4344   

                                            entry_coverage  entry_net  
model                        segment_value                             
forest_price                 higher                 0.1029    -0.0012  
                             lower                  0.0245    -0.0002  
forest_price_volume          higher                 0.0984     0.0008  
                             lower                  0.0405     0.0000  
logistic_price               higher                 0.2237     0.0003  
                             lower                  0.0814    -0.0008  
logistic_price_volume        higher                 0.2408     0.0001  
                             lower                  0.1033    -0.0008  
logistic_ticker_price        higher                 0.2543     0.0005  
                             lower                  0.0955    -0.0006  
logistic_ticker_price_volume higher                 0.2560     0.0001  
                             lower                  0.1216    -0.0001


 breadth_regime


n  accuracy  entry_precision  \
model                        segment_value                                     
forest_price                 advancing      83728    0.4209           0.4374   
                             declining      71993    0.4255           0.4490   
forest_price_volume          advancing      83728    0.4186           0.4388   
                             declining      71993    0.4254           0.4756   
logistic_price               advancing      83728    0.3997           0.4510   
                             declining      71993    0.4208           0.4444   
logistic_price_volume        advancing      83728    0.3989           0.4271   
                             declining      71993    0.4174           0.4483   
logistic_ticker_price        advancing      83728    0.3990           0.4461   
                             declining      71993    0.4175           0.4509   
logistic_ticker_price_volume advancing      83728    0.3983           0.4442   
                             declining      71993    0.4149           0.4450   

                                            entry_coverage  entry_net  
model                        segment_value                             
forest_price                 advancing              0.0321    -0.0014  
                             declining              0.0961    -0.0013  
forest_price_volume          advancing              0.0352     0.0003  
                             declining              0.1069     0.0003  
logistic_price               advancing              0.0941     0.0007  
                             declining              0.2101    -0.0008  
logistic_price_volume        advancing              0.1107    -0.0006  
                             declining              0.2333    -0.0004  
logistic_ticker_price        advancing              0.1140    -0.0000  
                             declining              0.2323     0.0000  
logistic_ticker_price_volume advancing              0.1342    -0.0000  
                             declining              0.2458    -0.0003


 momentum_regime


n  accuracy  entry_precision  \
model                        segment_value                                     
forest_price                 negative       68580    0.4231           0.4476   
                             positive       87141    0.4238           0.4428   
forest_price_volume          negative       68580    0.4242           0.4648   
                             positive       87141    0.4204           0.4709   
logistic_price               negative       68580    0.4085           0.4607   
                             positive       87141    0.4112           0.4281   
logistic_price_volume        negative       68580    0.4055           0.4481   
                             positive       87141    0.4099           0.4313   
logistic_ticker_price        negative       68580    0.4070           0.4617   
                             positive       87141    0.4085           0.4382   
logistic_ticker_price_volume negative       68580    0.4031           0.4497   
                             positive       87141    0.4091           0.4404   

                                            entry_coverage  entry_net  
model                        segment_value                             
forest_price                 negative               0.1013    -0.0012  
                             positive               0.0308    -0.0012  
forest_price_volume          negative               0.1199     0.0005  
                             positive               0.0279     0.0005  
logistic_price               negative               0.2017    -0.0000  
                             positive               0.1057    -0.0006  
logistic_price_volume        negative               0.2205    -0.0004  
                             positive               0.1258    -0.0006  
logistic_ticker_price        negative               0.2152     0.0004  
                             positive               0.1317    -0.0003  
logistic_ticker_price_volume negative               0.2500    -0.0001  
                             positive               0.1346    -0.0002

## 7. Probability calibration and confidence deciles

These deciles are ranked within each test fold. Look for monotonic increases in observed Up prevalence and net return, with meaningful sample counts. Do not interpret an extreme decile with few entries as validated.


In [8]:
dec=outputs['deciles']
display(dec.groupby(['model','decile']).agg(n=('n','sum'),p_up=('mean_p_up','mean'),
    observed_up=('observed_up','mean'),next_return=('mean_next_return','mean'),
    net_long=('mean_net_long','mean'),brier=('up_brier','mean')).round(4))


n    p_up  observed_up  next_return  \
model                        decile                                            
forest_price                 1       15573  0.3481       0.3882       0.0006   
                             2       15573  0.3799       0.3959       0.0002   
                             3       15573  0.3954       0.4087       0.0006   
                             4       15573  0.4079       0.4140       0.0006   
                             5       15573  0.4194       0.4170       0.0005   
                             6       15572  0.4308       0.4266       0.0007   
                             7       15572  0.4426       0.4341       0.0007   
                             8       15571  0.4562       0.4310       0.0007   
                             9       15571  0.4737       0.4413       0.0008   
                             10      15570  0.5124       0.4391       0.0002   
forest_price_volume          1       15573  0.3434       0.3834       0.0004   
                             2       15573  0.3783       0.3976       0.0003   
                             3       15573  0.3950       0.4156       0.0007   
                             4       15573  0.4079       0.4148       0.0005   
                             5       15573  0.4195       0.4150       0.0005   
                             6       15572  0.4308       0.4209       0.0003   
                             7       15572  0.4427       0.4262       0.0007   
                             8       15571  0.4564       0.4282       0.0006   
                             9       15571  0.4752       0.4426       0.0011   
                             10      15570  0.5181       0.4515       0.0006   
logistic_price               1       15573  0.3055       0.4327       0.0017   
                             2       15573  0.3558       0.3997       0.0005   
                             3       15573  0.3807       0.3892      -0.0004   
                             4       15573  0.3999       0.3852      -0.0007   
                             5       15573  0.4165       0.3998      -0.0002   
                             6       15572  0.4325       0.4154       0.0002   
                             7       15572  0.4497       0.4308       0.0009   
                             8       15571  0.4707       0.4405       0.0013   
                             9       15571  0.4995       0.4469       0.0014   
                             10      15570  0.5572       0.4556       0.0010   
logistic_price_volume        1       15573  0.2935       0.4222       0.0013   
                             2       15573  0.3467       0.4134       0.0008   
                             3       15573  0.3728       0.4083       0.0005   
                             4       15573  0.3942       0.4040       0.0002   
                             5       15573  0.4137       0.4024      -0.0001   
                             6       15572  0.4329       0.4149       0.0005   
                             7       15572  0.4533       0.4172       0.0004   
                             8       15571  0.4770       0.4228       0.0004   
                             9       15571  0.5079       0.4338       0.0006   
                             10      15570  0.5690       0.4567       0.0010   
logistic_ticker_price        1       15573  0.2983       0.4245       0.0014   
                             2       15573  0.3526       0.4101       0.0004   
                             3       15573  0.3786       0.3981       0.0002   
                             4       15573  0.3994       0.4095       0.0002   
                             5       15573  0.4182       0.4061       0.0003   
                             6       15572  0.4366       0.4043      -0.0001   
                             7       15572  0.4557       0.4142       0.0003   
                             8       15571  0.4778       0.4289       0.0008   
                             9       1557

## 8. Economic target: a distinct prediction task

Binary outcome: next return exceeds the assumed **10 bps** cost. Report its own majority baseline and ROC-AUC. Accuracy across this binary task and Q7 cannot be directly compared. The 10 bps cost is a simplifying assumption, not a calibrated estimate of spread, slippage, and market impact.


In [9]:
display(outputs['economic_targets'].round(4))


,fold,model,positive_base,binary_accuracy,binary_majority_baseline,binary_auc,selected_count,selected_mean_net
0,1,economic_binary_logistic,0.4819,0.5017,0.5181,0.5007,12503,0.0001
1,2,economic_binary_logistic,0.4904,0.5141,0.5096,0.5122,9531,0.0005
2,3,economic_binary_logistic,0.4974,0.5026,0.5026,0.5046,6528,-0.0022


## 9. Confusion matrices and compute budget


In [10]:
conf=outputs['confusions']
for name in sorted(conf.model.unique()):
    print(name)
    display(conf[conf.model.eq(name)].pivot_table(index='true_class',columns='pred_class',values='n',aggfunc='sum',fill_value=0))
display(outputs['timings'].groupby('model').agg(total_fit_seconds=('train_seconds','sum'),mean_train_rows=('train_rows','mean')).round(2))


forest_price


pred_class,0,1,2
true_class,,,
0,10362,36056,843
1,9274,54036,2028
2,5068,36507,1547


forest_price_volume


pred_class,0,1,2
true_class,,,
0,9949,36263,1049
1,8995,53822,2521
2,4807,36356,1959


logistic_price


pred_class,0,1,2
true_class,,,
0,9639,34747,2875
1,9046,50252,6040
2,4774,34415,3933


... (3 outputs left out)

pred_class,0,1,2
true_class,,,
0,10810,33170,3281
1,10385,48254,6699
2,5459,33236,4427


logistic_ticker_price_volume


pred_class,0,1,2
true_class,,,
0,11776,31608,3877
1,11503,46378,7457
2,6140,31865,5117


,total_fit_seconds,mean_train_rows
model,,
forest_price,5.38,35000.0
forest_price_volume,7.10,35000.0
logistic_price,16.06,50000.0
logistic_price_volume,45.33,50000.0
logistic_ticker_price,79.92,50000.0
logistic_ticker_price_volume,90.05,50000.0


## 10. Decision criteria for Notebook 28

- **Sector specialization:** pursue only if a real sector mapping exists and gains recur across folds with adequate sector sample sizes.
- **Ticker personalization:** require paired improvement over pooled price models and stability; ticker-specific leaderboard winners are exploratory.
- **Volume:** retain only when improvements recur in at least two of three folds and have nontrivial magnitude.
- **Economic target:** compare against its **own** majority baseline and examine calibration, coverage, and returns.
- **Entry policy:** no promotion from this notebook. A policy must be frozen before an independent prospective shadow evaluation.

**Do not reopen the historical holdout or modify the prospective prediction ledger.**
